# PointNet Refactor

In [1]:
from typing import Optional, Tuple, List, Union, Sequence
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_scatter import scatter_max, scatter_mean
# from torch_geometric.typing import OptTensor

In [2]:
class PackedTNet(nn.Module):
    """Input/Feature Transform Network for packed points"""
    def __init__(
        self,
        k: int,
        dims: Sequence[int] = (64, 128, 1024),
        mlp_dims: Sequence[int] = (512, 256)
    ) -> None:
        super().__init__()
        self.k = k
        
        # Point-wise MLPs
        layers = []
        prev_dim = k
        for dim in dims:
            layers.extend([
                nn.Linear(prev_dim, dim),
                nn.BatchNorm1d(dim),
                nn.ReLU()
            ])
            prev_dim = dim
        self.point_net = nn.Sequential(*layers)
        
        # Global MLP
        mlp_layers = []
        prev_dim = dims[-1]
        for dim in mlp_dims:
            mlp_layers.extend([
                nn.Linear(prev_dim, dim),
                nn.BatchNorm1d(dim),
                nn.ReLU()
            ])
            prev_dim = dim
        mlp_layers.append(nn.Linear(prev_dim, k * k))
        self.mlp = nn.Sequential(*mlp_layers)
        
    def forward(self, x: torch.Tensor, batch: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (N, K) Packed point features
            batch: (N,) Batch assignment for each point
        Returns:
            (B, K, K) Transformation matrices for each cloud
        """
        num_clouds = batch.max().item() + 1
        
        # Point-wise feature extraction
        feat = self.point_net(x)
        
        # Global max pooling per cloud
        global_feat = scatter_max(feat, batch, dim=0)[0]  # (B, dims[-1])
        
        # Generate transformation matrix per cloud
        x = self.mlp(global_feat)  # (B, K*K)
        
        identity = torch.eye(self.k, requires_grad=True, device=x.device)
        identity = identity.repeat(num_clouds, 1, 1)
        
        x = x.view(-1, self.k, self.k) + identity
        return x

In [3]:
class PackedPointNetEncoder(nn.Module):
    """PointNet Encoder for packed points"""
    def __init__(
        self,
        in_channels: int,
        channel_dims: Sequence[int] = (64, 128, 1024),
        use_transform: bool = True,
        tnet_dims: Optional[Sequence[int]] = None,
        tnet_mlp_dims: Optional[Sequence[int]] = None
    ) -> None:
        super().__init__()
        
        self.use_transform = use_transform
        if use_transform:
            self.input_transform = PackedTNet(
                k=in_channels,
                dims=tnet_dims or (64, 128, 1024),
                mlp_dims=tnet_mlp_dims or (512, 256)
            )
            self.feature_transform = PackedTNet(
                k=channel_dims[0],
                dims=tnet_dims or (64, 128, 1024),
                mlp_dims=tnet_mlp_dims or (512, 256)
            )
        
        # Build point-wise MLPs
        layers = []
        prev_dim = in_channels
        for dim in channel_dims:
            layers.extend([
                nn.Linear(prev_dim, dim),
                nn.BatchNorm1d(dim),
                nn.ReLU()
            ])
            prev_dim = dim
        
        self.point_net = nn.Sequential(*layers)
        self.out_features = channel_dims[-1]
            
    def apply_transform(self, x: torch.Tensor, transform: torch.Tensor, batch: torch.Tensor) -> torch.Tensor:
        """
        Apply transformation to packed points
        Args:
            x: (N, C) Input features
            transform: (B, K, K) Transformation matrices
            batch: (N,) Batch assignments
        Returns:
            (N, C) Transformed features
        """
        B, K, _ = transform.shape
        N = x.size(0)
        
        # Expand transform to match points
        # (N, K, K)
        transform_expanded = transform[batch]
        
        # Reshape input to (N, 1, C)
        x_reshaped = x.unsqueeze(1)
        
        # Apply transformation: (N, 1, K) @ (N, K, K) -> (N, 1, K)
        x_transformed = torch.bmm(x_reshaped, transform_expanded)
        
        # Reshape back to (N, C)
        return x_transformed.squeeze(1)
    
    def forward(
        self, 
        x: torch.Tensor,  # (N, C)
        batch: torch.Tensor,  # (N,)
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        """
        Args:
            x: (N, C) Packed point features
            batch: (N,) Batch assignment for each point
        Returns:
            features: (B, C) Global features per cloud
            transform: Optional[Tensor] Feature transform matrix if use_transform=True
        """
        transform_matrix = None
        
        if self.use_transform:
            # Apply input transform per cloud
            input_transform = self.input_transform(x, batch)  # (B, K, K)
            x = self.apply_transform(x, input_transform, batch)
        
        # First MLP layer
        x = self.point_net[:3](x)
            
        if self.use_transform:
            # Apply feature transform per cloud
            transform_matrix = self.feature_transform(x, batch)
            x = self.apply_transform(x, transform_matrix, batch)
            
        # Remaining MLP layers
        x = self.point_net[3:](x)
            
        # Global max pooling per cloud
        x = scatter_max(x, batch, dim=0)[0]
        
        return x, transform_matrix

In [4]:
class PackedPointNet(nn.Module):
    """PointNet for packed point clouds"""
    def __init__(
        self,
        in_channels: int = 3,
        encoder_dims: Sequence[int] = (64, 128, 1024),
        mlp_dims: Sequence[int] = (512, 256),
        num_classes: int = 40,
        dropout: float = 0.3,
        use_transform: bool = True,
        tnet_dims: Optional[Sequence[int]] = None,
        tnet_mlp_dims: Optional[Sequence[int]] = None
    ) -> None:
        super().__init__()
        
        self.encoder = PackedPointNetEncoder(
            in_channels=in_channels,
            channel_dims=encoder_dims,
            use_transform=use_transform,
            tnet_dims=tnet_dims,
            tnet_mlp_dims=tnet_mlp_dims
        )
        
        # Build classifier MLP
        classifier_layers = []
        prev_dim = encoder_dims[-1]
        for dim in mlp_dims:
            classifier_layers.extend([
                nn.Linear(prev_dim, dim),
                nn.BatchNorm1d(dim),
                nn.ReLU(),
                nn.Dropout(p=dropout)
            ])
            prev_dim = dim
        classifier_layers.append(nn.Linear(prev_dim, num_classes))
        
        self.classifier = nn.Sequential(*classifier_layers)
    
    def forward(
        self,
        x: torch.Tensor,  # (N, C)
        batch: torch.Tensor,  # (N,)
        return_features: bool = False
    ) -> Union[torch.Tensor, Tuple[torch.Tensor, torch.Tensor, Optional[torch.Tensor]]]:
        """
        Args:
            x: (N, C) Packed point features
            batch: (N,) Batch assignment for each point
            return_features: Whether to return intermediate features
        Returns:
            output: (B, num_classes) Classification logits
            features: Optional[(B, C)] Global features if return_features=True
            transform: Optional[(B, K, K)] Transform matrix if return_features=True
        """
        features, transform = self.encoder(x, batch)
        output = self.classifier(features)
        
        if return_features:
            return output, features, transform
        return output
    
    def get_regularization_loss(self, transform_matrix: torch.Tensor) -> torch.Tensor:
        """Compute regularization loss for the feature transform matrix"""
        identity = torch.eye(
            transform_matrix.size(-1),
            requires_grad=True,
            device=transform_matrix.device
        ).unsqueeze(0)
        
        loss = torch.mean(
            torch.norm(
                torch.bmm(transform_matrix, transform_matrix.transpose(2, 1)) - identity,
                dim=(1, 2)
            )
        )
        return loss

In [5]:
# Pre-defined model configurations
def packed_pointnet_small(num_classes: int = 40, **kwargs) -> PackedPointNet:
    return PackedPointNet(
        encoder_dims=(32, 64, 512),
        mlp_dims=(256,),
        num_classes=num_classes,
        **kwargs
    )

def packed_pointnet_vanilla(num_classes: int = 40, **kwargs) -> PackedPointNet:
    return PackedPointNet(
        encoder_dims=(64, 128, 1024),
        mlp_dims=(512, 256),
        num_classes=num_classes,
        **kwargs
    )

def packed_pointnet_large(num_classes: int = 40, **kwargs) -> PackedPointNet:
    return PackedPointNet(
        encoder_dims=(64, 128, 256, 512, 2048),
        mlp_dims=(1024, 512),
        num_classes=num_classes,
        **kwargs
    )

In [9]:
# Create model
model = PackedPointNet(in_channels=7, num_classes=40)

# Sample data
points = torch.randn(1000, 7)  # 1000 points total
batch = torch.tensor([0] * 400 + [1] * 600)  # 2 point clouds (400 + 600 points)

# Forward pass
output = model(points, batch)
output.shape

torch.Size([2, 40])

In [7]:
output

tensor([[-0.2147, -0.2680,  0.2827,  0.6020, -0.0187, -1.2293, -0.2454, -0.8031,
          0.5990,  0.7923, -0.0944, -0.1219,  1.5567, -0.6394, -0.4627, -0.1746,
          0.0530, -0.2126,  0.0109,  0.4201,  0.7986, -0.1479, -0.7034,  0.0811,
         -0.3964,  0.3868, -0.1236,  0.6470,  0.2526,  0.1442, -0.3602, -0.0915,
          0.0101,  0.5538,  0.0733, -0.3710,  0.0017,  0.0901,  0.0326,  0.9582],
        [-0.2563, -0.1088,  0.7764,  0.4062,  0.1149,  0.9799, -0.1844, -0.1933,
         -0.0187, -0.1753, -0.6764, -0.3471, -0.4969,  0.2520, -0.1769, -1.3559,
          0.7044,  0.2438, -0.6476, -0.8311, -0.7991,  0.7554,  0.2911, -0.7193,
         -0.2189,  0.1477,  0.0910,  0.2858,  0.1185,  0.3530,  0.1288,  0.9046,
         -0.2604, -1.1139, -0.0464,  0.2931, -0.7384, -0.0722,  0.4675, -0.6622]],
       grad_fn=<AddmmBackward0>)

---

In [1]:
from typing import Optional, Tuple, List, Union, Sequence
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_scatter import scatter_max, scatter_mean

In [2]:
class PackedTNet(nn.Module):
    """Input/Feature Transform Network for packed points"""
    def __init__(
        self,
        k: int,
        dims: Sequence[int] = (64, 128, 1024),
        mlp_dims: Sequence[int] = (512, 256)
    ) -> None:
        super().__init__()
        self.k = k
        
        # Point-wise MLPs
        layers = []
        prev_dim = k
        for dim in dims:
            layers.extend([
                nn.Linear(prev_dim, dim),
                nn.BatchNorm1d(dim),
                nn.ReLU()
            ])
            prev_dim = dim
        self.point_net = nn.Sequential(*layers)
        
        # Global MLP
        mlp_layers = []
        prev_dim = dims[-1]
        for dim in mlp_dims:
            mlp_layers.extend([
                nn.Linear(prev_dim, dim),
                nn.BatchNorm1d(dim),
                nn.ReLU()
            ])
            prev_dim = dim
        mlp_layers.append(nn.Linear(prev_dim, k * k))
        self.mlp = nn.Sequential(*mlp_layers)
        
    def forward(self, x: torch.Tensor, batch: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (N, K) Packed point features
            batch: (N,) Batch assignment for each point
        Returns:
            (B, K, K) Transformation matrices for each cloud
        """
        # Point-wise feature extraction
        feat = self.point_net(x)
        
        # Global max pooling per cloud
        global_feat = scatter_max(feat, batch, dim=0)[0]
        
        # Generate transformation matrix per cloud
        x = self.mlp(global_feat)
        
        # Add identity matrix
        identity = torch.eye(
            self.k,
            requires_grad=True,
            device=x.device
        ).unsqueeze(0).repeat(batch.max().item() + 1, 1, 1)
        
        x = x.view(-1, self.k, self.k) + identity
        return x

class PackedPointNetEncoder(nn.Module):
    """PointNet Encoder with separate handling of coordinates and features"""
    def __init__(
        self,
        coord_dim: int = 3,
        feat_dim: int = 0,
        channel_dims: Sequence[int] = (64, 128, 1024),
        use_spatial_transform: bool = True,
        use_feature_transform: bool = False,
        tnet_dims: Optional[Sequence[int]] = None,
        tnet_mlp_dims: Optional[Sequence[int]] = None
    ) -> None:
        super().__init__()
        
        self.coord_dim = coord_dim
        self.feat_dim = feat_dim
        self.use_spatial_transform = use_spatial_transform
        self.use_feature_transform = use_feature_transform
        
        # Spatial transform for coordinates
        if use_spatial_transform:
            self.spatial_transform = PackedTNet(
                k=coord_dim,
                dims=tnet_dims or (64, 128, 1024),
                mlp_dims=tnet_mlp_dims or (512, 256)
            )
            
        # First layer processes coordinates and features separately
        self.coord_layer = nn.Sequential(
            nn.Linear(coord_dim, channel_dims[0] // 2),
            nn.BatchNorm1d(channel_dims[0] // 2),
            nn.ReLU()
        )
        
        if feat_dim > 0:
            self.feat_layer = nn.Sequential(
                nn.Linear(feat_dim, channel_dims[0] // 2),
                nn.BatchNorm1d(channel_dims[0] // 2),
                nn.ReLU()
            )
        
        # Feature transform after first layer
        if use_feature_transform:
            self.feature_transform = PackedTNet(
                k=channel_dims[0],
                dims=tnet_dims or (64, 128, 1024),
                mlp_dims=tnet_mlp_dims or (512, 256)
            )
        
        # Remaining point-wise MLPs
        layers = []
        prev_dim = channel_dims[0]
        for dim in channel_dims[1:]:
            layers.extend([
                nn.Linear(prev_dim, dim),
                nn.BatchNorm1d(dim),
                nn.ReLU()
            ])
            prev_dim = dim
        
        self.point_net = nn.Sequential(*layers)
        self.out_features = channel_dims[-1]
    
    def apply_transform(self, x: torch.Tensor, transform: torch.Tensor, batch: torch.Tensor) -> torch.Tensor:
        """Apply transformation to packed points"""
        transform_expanded = transform[batch]
        x_reshaped = x.unsqueeze(1)
        x_transformed = torch.bmm(x_reshaped, transform_expanded)
        return x_transformed.squeeze(1)
    
    def forward(
        self, 
        coords: torch.Tensor,  # (N, 3)
        batch: torch.Tensor,  # (N,)
        features: Optional[torch.Tensor] = None,  # (N, F)
    ) -> Tuple[torch.Tensor, Tuple[Optional[torch.Tensor], Optional[torch.Tensor]]]:
        """
        Args:
            coords: (N, 3) Packed point coordinates
            batch: (N,) Batch assignment for each point
            features: Optional[Tensor] (N, F) Additional point features
        Returns:
            features: (B, C) Global features per cloud
            transforms: Tuple[Optional[Tensor], Optional[Tensor]] Spatial and feature transforms
        """
        spatial_transform = None
        feature_transform = None
        
        # Apply spatial transform to coordinates
        if self.use_spatial_transform:
            spatial_transform = self.spatial_transform(coords, batch)
            coords = self.apply_transform(coords, spatial_transform, batch)
        
        # Process coordinates
        x = self.coord_layer(coords)
        
        # Process and concatenate features if present
        if self.feat_dim > 0 and features is not None:
            feat = self.feat_layer(features)
            x = torch.cat([x, feat], dim=1)
            
        # Apply feature transform
        if self.use_feature_transform:
            feature_transform = self.feature_transform(x, batch)
            x = self.apply_transform(x, feature_transform, batch)
            
        # Remaining layers
        x = self.point_net(x)
            
        # Global max pooling per cloud
        x = scatter_max(x, batch, dim=0)[0]
        
        return x, (spatial_transform, feature_transform)

class PackedPointNet(nn.Module):
    """PointNet with separate coordinate and feature handling"""
    def __init__(
        self,
        coord_dim: int = 3,
        feat_dim: int = 0,
        encoder_dims: Sequence[int] = (64, 128, 1024),
        mlp_dims: Sequence[int] = (512, 256),
        num_classes: int = 40,
        dropout: float = 0.3,
        use_spatial_transform: bool = True,
        use_feature_transform: bool = False,
        tnet_dims: Optional[Sequence[int]] = None,
        tnet_mlp_dims: Optional[Sequence[int]] = None
    ) -> None:
        super().__init__()
        
        self.encoder = PackedPointNetEncoder(
            coord_dim=coord_dim,
            feat_dim=feat_dim,
            channel_dims=encoder_dims,
            use_spatial_transform=use_spatial_transform,
            use_feature_transform=use_feature_transform,
            tnet_dims=tnet_dims,
            tnet_mlp_dims=tnet_mlp_dims
        )
        
        # Build classifier MLP
        classifier_layers = []
        prev_dim = encoder_dims[-1]
        for dim in mlp_dims:
            classifier_layers.extend([
                nn.Linear(prev_dim, dim),
                nn.BatchNorm1d(dim),
                nn.ReLU(),
                nn.Dropout(p=dropout)
            ])
            prev_dim = dim
        classifier_layers.append(nn.Linear(prev_dim, num_classes))
        
        self.classifier = nn.Sequential(*classifier_layers)
    
    def _init_weights(self) -> None:
        """Initialize weights following modern practices"""
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def forward(
        self,
        coords: torch.Tensor,  # (N, 3)
        batch: torch.Tensor,  # (N,)
        features: Optional[torch.Tensor] = None,  # (N, F)
        return_features: bool = False
    ) -> Union[
        torch.Tensor,
        Tuple[torch.Tensor, torch.Tensor, Tuple[Optional[torch.Tensor], Optional[torch.Tensor]]]
    ]:
        """
        Args:
            coords: (N, 3) Packed point coordinates
            batch: (N,) Batch assignment for each point
            features: Optional[Tensor] (N, F) Additional point features
            return_features: Whether to return intermediate features
        """
        global_features, transforms = self.encoder(coords, batch, features)
        output = self.classifier(global_features)
        
        if return_features:
            return output, global_features, transforms
        return output

In [3]:
model = PackedPointNet(
    coord_dim=3,           # XYZ coordinates
    feat_dim=3,           # RGB or normal features
    use_spatial_transform=True,   # Transform coordinates
    use_feature_transform=False,  # Don't transform features
    num_classes=40
)

# Sample data
coords = torch.randn(1000, 3)     # XYZ coordinates
features = torch.randn(1000, 3)   # RGB or normal features
batch = torch.tensor([0] * 400 + [1] * 600)

# Forward pass
output = model(coords, batch, features)

In [4]:
output.shape

torch.Size([2, 40])

In [15]:
import timm


timm.create_model('resnet50', pretrained=True).fc

model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

Linear(in_features=2048, out_features=1000, bias=True)

In [14]:
from torch_geometric.nn import MLP


MLP([1024, 512, 256, 10], dropout=0.5, norm=None)

MLP(1024, 512, 256, 10)